# SSTW B5 Colab GPU 冷启动运行入口

该 Notebook 只作为 Colab 入口使用。正式 records、tables、reports、artifacts 与 package manifest 由仓库脚本和 `experiments/` 模块生成。

冷启动流程必须完成:

```text
挂载 Google Drive
clone / pull SSTW 仓库
安装依赖
准备 prompt suite
运行真实 Wan2.1 GPU 生成
运行 formal metric / pilot postprocess / runtime attack / runtime detection / external baseline adapter comparison / pilot gate / validation internal ablation / CI / FPR=0.01 pilot gate / frozen threshold artifact / artifact rebuild dry-run / validation-scale gate
打包到 /content/drive/MyDrive/SSTW/packages/generative_video_model_probe
```

当前默认用途是 `validation_scale`。该模式用于 pilot 通过后的 validation-scale 预演, 目标是用 8 个 prompt、每个 prompt 3 个 seed 验证生成、attack、detection、外部 baseline 状态与 adapter comparison、内部消融、adaptive attack、replay/sketch、CI 和 artifact rebuild 门禁是否闭合。Notebook 会先生成内部消融 proxy、统计 CI proxy 和 artifact rebuild dry-run 产物, 再由 validation-scale gate 汇总缺口。默认主模型为 `Wan-AI/Wan2.1-T2V-1.3B-Diffusers`。

注意: `pilot`、`validation_scale` 与 `fpr01_pilot` 模式都不会重新估计 motion threshold。若缺少已通过的 calibration artifact, 请先切回 `PROFILE = 'motion_calibration'` 完成阈值校准。`fpr01_pilot` 会使用 21 个 prompt × 8 个 seed, 并按 4 个 calibration seed 与 4 个 held-out test seed 执行固定阈值协议。


In [ ]:
# 1. 挂载 Google Drive 并检查 GPU
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, json
from pathlib import Path

print(subprocess.getoutput('nvidia-smi'))
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/SSTW'
Path(DRIVE_PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
print('drive_project_root =', DRIVE_PROJECT_ROOT)


In [ ]:
# 2. 获取仓库
# Colab 是冷启动环境, 默认从 GitHub 拉取最新 main 分支。
# 如果你使用私有 fork, 请把 REPO_URL 改成你自己的可访问地址。
REPO_URL = 'https://github.com/RICHAAARC/SSTW.git'
REPO_DIR = '/content/SSTW'

if not Path(REPO_DIR).exists():
    if not REPO_URL:
        raise RuntimeError('请先设置 REPO_URL, 或将仓库上传到 /content/SSTW')
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print('仓库目录已存在, 执行 git pull 以同步远端代码。')
    %cd "$REPO_DIR"
    !git pull --ff-only
%cd "$REPO_DIR"
!git rev-parse --short HEAD

In [ ]:
# 3. 安装 Colab GPU 运行依赖
%pip install -U git+https://github.com/huggingface/diffusers transformers accelerate safetensors imageio imageio-ffmpeg sentencepiece protobuf


In [ ]:
# 4. 可选 Hugging Face 认证
# 公开模型通常可以匿名下载, 但 Colab 中建议提供 HF_TOKEN 以避免限流或访问 gated 模型失败。
# 注意: token 只写入当前 Colab 环境变量, 不写入 records、tables, reports 或 package manifest。
import os
# The user wants to strictly use the environment's HF_TOKEN, so removing the interactive getpass.
# from getpass import getpass # This line is no longer needed.

try:
    from huggingface_hub import login
except Exception as exc:
    raise RuntimeError('huggingface_hub 未安装或无法导入, 请重新执行依赖安装单元') from exc

# Check if HF_TOKEN is set in the environment
if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF_TOKEN 状态: provided from environment.') # Clarified source
else:
    print('HF_TOKEN 状态: not_provided in environment, 将尝试匿名下载公开模型。')


In [ ]:
# 5. 初始化 Drive 目录布局
from paper_workflow.notebook_utils import generative_video_model_probe_workflow as probe_workflow

layout = probe_workflow.ensure_drive_layout(DRIVE_PROJECT_ROOT)
print(json.dumps(layout, ensure_ascii=False, indent=2))


In [ ]:
# 6. 构造 prompt / seed / motion 数据集
# 该步骤只构造测试输入数据, 不执行模型测试。输出落盘到 Google Drive datasets 子目录。
cmd = probe_workflow.build_prompt_suite_command(layout)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 7. 配置 B5 Colab 运行目标
# smoke: 最小冷启动验证。recommended: 更多 prompt / seed。pilot: small-scale claim pilot。validation_scale: pilot 通过后的 validation-scale 预演。fpr01_pilot: 支撑 pilot 级 TPR@FPR=0.01 报告的 21 prompt × 8 seed 中等规模预演, 其中 4 个 calibration seed 冻结阈值, 4 个 test seed 报告 held-out FPR / TPR。motion_calibration: 128/64/32 motion threshold calibration split。extended: 更长运行。
PROFILE = 'validation_scale'
MODEL_ID = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
# cross model 必须兼容当前 colab_runtime 的 LTXPipeline 调用。若不确定, 保持为空, 记录为 not_configured。
CROSS_MODEL_ID = ''
# 正式语义 metric 使用 CLIP 文本-视频相似度。该模型为公开模型, 通常不强制 HF_TOKEN, 但提供 token 可降低限流风险。
SEMANTIC_MODEL_ID = 'openai/clip-vit-base-patch32'
SEMANTIC_FRAME_LIMIT = 8
DISABLE_SEMANTIC_METRIC = False
INCLUDE_VIDEOS_IN_PACKAGE = True


In [ ]:
# 8. 执行真实 GPU 生成测试
# 正式 records / tables / artifacts 会由 experiments 模块写入 Drive run 子目录。
cmd = probe_workflow.build_colab_runtime_command(layout, PROFILE, MODEL_ID, CROSS_MODEL_ID)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 9. 执行正式质量、运动与语义度量
# 该步骤读取已生成 mp4 文件和 prompt suite, 写出 formal_quality_motion_semantic_records。
# 如果 CLIP 依赖或模型权重不可用, records 会显式保留阻断原因, 不伪造语义通过。
cmd = probe_workflow.build_formal_metric_command(
    layout,
    semantic_model_id=SEMANTIC_MODEL_ID,
    semantic_frame_limit=SEMANTIC_FRAME_LIMIT,
    disable_semantic_metric=DISABLE_SEMANTIC_METRIC,
)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 10. 执行或复用 motion threshold calibration
# motion_calibration profile 会生成并冻结 128 / 64 / 32 calibration threshold。
# pilot / validation_scale profile 只能复用已经通过的 calibration artifact, 不能用 evaluation 样本重新估计阈值。
from pathlib import Path
import json

if PROFILE == 'motion_calibration':
    cmd = probe_workflow.build_motion_threshold_calibration_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)
else:
    decision_path = Path(layout['drive_run_root']) / 'artifacts' / 'motion_threshold_calibration_decision.json'
    if not decision_path.exists():
        raise RuntimeError(
            'pilot 或 validation_scale profile 需要先存在已通过的 motion_threshold_calibration_decision.json。'
            '请先运行 PROFILE=motion_calibration, 或确认 Google Drive run_root 中保留了已通过阈值 artifact。'
        )
    motion_threshold_decision = json.loads(decision_path.read_text(encoding='utf-8-sig'))
    if motion_threshold_decision.get('motion_threshold_calibration_ready') is not True:
        raise RuntimeError(
            'pilot 或 validation_scale profile 不能使用未通过的 motion threshold calibration artifact: '
            + str(motion_threshold_decision.get('motion_threshold_calibration_decision'))
        )
    print('复用已通过的 motion threshold calibration artifact:')
    print(json.dumps({
        'motion_threshold_calibration_decision': motion_threshold_decision.get('motion_threshold_calibration_decision'),
        'motion_threshold_id': motion_threshold_decision.get('motion_threshold_id'),
        'motion_threshold_source_split': motion_threshold_decision.get('motion_threshold_source_split'),
        'motion_delta_threshold': motion_threshold_decision.get('motion_delta_threshold'),
        'claim_support_status': motion_threshold_decision.get('claim_support_status'),
    }, ensure_ascii=False, indent=2))


In [ ]:
# 11. 执行机制后处理
# motion_calibration 只需要校准 motion threshold, 不需要生成 small-scale pilot 后处理证据。
if PROFILE == 'motion_calibration':
    print('PROFILE=motion_calibration, 跳过 mechanism postprocess。')
else:
    cmd = probe_workflow.build_mechanism_postprocess_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)

In [ ]:
# 12. 补齐 small-scale pilot 实验矩阵后处理
# motion_calibration split 不能混用为 small-scale pilot matrix。
if PROFILE == 'motion_calibration':
    print('PROFILE=motion_calibration, 跳过 pilot matrix postprocess。')
else:
    cmd = probe_workflow.build_pilot_matrix_postprocess_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)

In [ ]:
# 13. 执行真实 runtime 文件级攻击
# motion_calibration 阶段不需要对 224 个 calibration videos 生成 attacked videos, 否则会显著放大运行时间和 Drive 占用。
if PROFILE == 'motion_calibration':
    print('PROFILE=motion_calibration, 跳过 runtime attack。')
else:
    cmd = probe_workflow.build_runtime_attack_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)

In [ ]:
# 14. 执行 runtime attacked video 检测评分
# 仅当 runtime attack 已执行时才需要该步骤。
if PROFILE == 'motion_calibration':
    print('PROFILE=motion_calibration, 跳过 runtime detection。')
else:
    cmd = probe_workflow.build_runtime_detection_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)

In [ ]:
# 15. 构建 external_baseline adapter 对比结果
# 该步骤从 runtime detection records 和 callback trajectory records 生成 baseline comparison records。
# 显式 DTW 与 frame matching 当前是工程级同步 control, modern video watermark baseline 未接入前会写为 unsupported row。
if PROFILE == 'motion_calibration':
    print('PROFILE=motion_calibration, 跳过 external baseline comparison。')
else:
    cmd = probe_workflow.build_external_baseline_comparison_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)


In [ ]:
# 16. 写出 small-scale claim pilot gate
# motion_calibration 只产出 threshold calibration artifact, 不产出 small-scale claim pilot gate。
if PROFILE == 'motion_calibration':
    print('PROFILE=motion_calibration, 跳过 small-scale claim pilot gate。')
else:
    cmd = probe_workflow.build_small_scale_claim_pilot_gate_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)

In [ ]:
# 17. 构建 validation-scale 内部消融矩阵
# 该步骤只基于 runtime detection proxy records 生成 validation 级消融 records, 不生成 full-paper 消融表。
if PROFILE == 'motion_calibration':
    print('PROFILE=motion_calibration, 跳过 validation internal ablation。')
else:
    cmd = probe_workflow.build_validation_internal_ablation_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)


In [ ]:
# 18. 构建 validation-scale adaptive attack proxy records
# 该步骤只从 runtime detection records 写出 Flow-specific adaptive attack proxy, 不支撑 full-paper adaptive robustness claim。
if PROFILE == 'motion_calibration':
    print('PROFILE=motion_calibration, 跳过 adaptive attack validation proxy。')
else:
    cmd = probe_workflow.build_adaptive_attack_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)


In [ ]:
# 19. 构建 replay/sketch gate validation proxy records
# 该步骤只从 generation records 与 trajectory trace 写出 authenticated sketch / replay controls, 不支撑 full-paper 强 Claim-3。
if PROFILE == 'motion_calibration':
    print('PROFILE=motion_calibration, 跳过 replay/sketch gate。')
else:
    cmd = probe_workflow.build_replay_and_sketch_gate_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('replay/sketch gate 失败')


In [ ]:
# 20. 写出 Claim-3 downgrade gate
# 该步骤在 replay/sketch gate 未闭合时显式降级 Claim-3, 使 validation-scale 可合规继续。
if PROFILE == 'motion_calibration':
    print('PROFILE=motion_calibration, 跳过 Claim-3 downgrade gate。')
else:
    cmd = probe_workflow.build_claim3_downgrade_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)


In [ ]:
# 21. 构建 validation-scale 统计置信区间报告
# 该步骤只计算 validation runtime detection proxy 的轻量 Wilson 区间, 不能替代 full-paper FPR=0.001 统计报告。
if PROFILE == 'motion_calibration':
    print('PROFILE=motion_calibration, 跳过 statistical confidence interval reporter。')
else:
    cmd = probe_workflow.build_statistical_confidence_interval_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)


In [ ]:
# 22. 写出 FPR=0.01 pilot gate
# 该 gate 只在 PROFILE=fpr01_pilot 时运行, 用于支撑 pilot 级 TPR@FPR=0.01 报告。
# 它强制使用 calibration split 冻结阈值, 再用 held-out test split 报告 FPR / TPR。
# 它不允许输出 TPR@FPR=0.001 或 full-paper 主表结论。
if PROFILE == 'fpr01_pilot':
    cmd = probe_workflow.build_fpr01_pilot_gate_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)
else:
    print(f'PROFILE={PROFILE}, 跳过 FPR=0.01 pilot gate。')


In [ ]:
# 22. 执行 validation-scale artifact rebuild dry-run
# 该步骤只检查 records、tables、reports 和 artifacts 是否具备重建闭环, 不生成 full-paper package。
if PROFILE == 'motion_calibration':
    print('PROFILE=motion_calibration, 跳过 validation artifact rebuild dry-run。')
else:
    cmd = probe_workflow.build_validation_artifact_rebuild_dry_run_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)


In [ ]:
# 23. 写出 validation-scale gate
# 该 gate 不生成 full_paper 结果, 只审计 validation-scale 是否具备进入 full_paper dry-run checker 的条件。
if PROFILE in {'motion_calibration', 'fpr01_pilot'}:
    print('当前 PROFILE 不需要运行 validation-scale gate, 已跳过。')
else:
    cmd = probe_workflow.build_validation_scale_gate_command(layout)
    print(' '.join(cmd))
    result = probe_workflow.run_command(cmd)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)


In [ ]:
# 24. 运行仓库 quick tests 与 harness 审计
# 注意: pytest 默认不运行重型 GPU 测试。这里检查治理约束和轻量功能测试。
!pytest -q
!python tools/harness/run_all_audits.py


In [ ]:
# 24. 打包并落盘到 Google Drive packages 子目录
# 打包逻辑由 scripts/package_results/generative_video_drive_packager.py 执行, Notebook 不直接写正式 package manifest。
cmd = probe_workflow.build_drive_packaging_command(layout, include_videos=INCLUDE_VIDEOS_IN_PACKAGE)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 24. 显示可下载审阅文件
package_dir = Path(layout['drive_package_dir'])
print('package_dir =', package_dir)
for path in sorted(package_dir.glob('*')):
    print(path)
